# 2026 WBC

In [1]:
import requests

def get_wbc_schedule():
    # The MLB Stats API uses sportId=51 for International Baseball
    url = "https://statsapi.mlb.com/api/v1/schedule"
    
    # The 2026 WBC runs from March 4th through March 17th
    params = {
        "sportId": 51,
        "startDate": "2026-03-04",
        "endDate": "2026-03-17"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        for date_data in data.get('dates', []):
            print(f"\n--- Games on {date_data['date']} ---")
            
            for game in date_data.get('games', []):
                away_team = game['teams']['away']['team']['name']
                home_team = game['teams']['home']['team']['name']
                game_pk = game['gamePk'] # Crucial ID for getting live play-by-play later
                
                print(f"{away_team} vs {home_team} (Game ID: {game_pk})")
                
                # Flagging specific games 
                if "Mexico" in [away_team, home_team]:
                    print("   -> 🇲🇽 Team Mexico matchup")
    else:
        print(f"Error fetching data: {response.status_code}")

if __name__ == "__main__":
    get_wbc_schedule()


--- Games on 2026-03-04 ---
Canada vs Philadelphia Phillies (Game ID: 831427)
Colombia vs Atlanta Braves (Game ID: 831438)
Kingdom of the Netherlands vs Tampa Bay Rays (Game ID: 831432)
Nicaragua vs St. Louis Cardinals (Game ID: 831429)
Panama vs Detroit Tigers (Game ID: 831433)
Puerto Rico vs Minnesota Twins (Game ID: 831435)
Israel vs New York Mets (Game ID: 831431)
Detroit Tigers vs Dominican Republic (Game ID: 836150)
Brazil vs Texas Rangers (Game ID: 831938)
Cuba vs Cincinnati Reds (Game ID: 831852)
Mexico vs Los Angeles Dodgers (Game ID: 831799)
   -> 🇲🇽 Team Mexico matchup
Great Britain vs San Diego Padres (Game ID: 831854)
Italy vs Los Angeles Angels (Game ID: 831844)
United States vs Colorado Rockies (Game ID: 831723)
Venezuela vs Washington Nationals (Game ID: 831428)

--- Games on 2026-03-05 ---
Chinese Taipei vs Australia (Game ID: 788120)
Czechia vs Korea (Game ID: 788115)

--- Games on 2026-03-06 ---
Australia vs Czechia (Game ID: 788116)
Japan vs Chinese Taipei (Game ID

# International Leagues

In [3]:
import requests
import pandas as pd

def fetch_all_mlb_leagues():
    url = "https://statsapi.mlb.com/api/v1/leagues"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        leagues = data.get('leagues', [])
        
        # Create a list of dictionaries for easy processing
        league_list = []
        for l in leagues:
            league_list.append({
                'ID': l.get('id'),
                'Name': l.get('name'),
                'Abbreviation': l.get('abbreviation'),
                'Sport_ID': l.get('sport', {}).get('id')
            })
        
        return pd.DataFrame(league_list)
    else:
        return f"Error: {response.status_code}"

df = fetch_all_mlb_leagues()

# Filter for International (Sport ID 51) to find WBC, Premier 12, etc.
intl_leagues = df[df['Sport_ID'] == 51]
print(intl_leagues)

     ID                                  Name Abbreviation  Sport_ID
81  508   International Baseball (Collegiate)         INTC      51.0
82  551            China Baseball Association          CBA      51.0
83  552  Chinese Professional Baseball League         CPBL      51.0
84  580             International Competition          INT      51.0
87  159     World Baseball Classic Qualifiers         WBCQ      51.0
88  160                World Baseball Classic          WBC      51.0
89  475              International Exhibition          EXH      51.0
90  420                 Japan All-Star Series          JAS      51.0
91  515    USA vs. Canada Olympic Exhibitions           OE      51.0


# International Champions, incluiding other than WBC tournaments

In [5]:
import requests
import os

def get_wbc_champions():
    # Fetch seasons specifically for the WBC (League 160)
    seasons_url = "https://statsapi.mlb.com/api/v1/seasons/all?sportId=51&leagueId=160"
    response = requests.get(seasons_url)
    
    if response.status_code != 200:
        print("Error fetching seasons data.")
        return

    seasons = [s['seasonId'] for s in response.json().get('seasons', [])]
    
    print("World Baseball Classic Champions:")
    print("-" * 65)

    for year in sorted(seasons):
        # Filter schedule by both Sport ID and League ID
        schedule_url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=51&leagueId=160&season={year}"
        sched_response = requests.get(schedule_url).json()
        
        dates = sched_response.get('dates', [])
        if not dates:
            continue
            
        final_date = dates[-1]
        final_game = final_date.get('games', [])[-1]
        
        away = final_game['teams']['away']
        home = final_game['teams']['home']
        
        # Since we filtered by leagueId=160, we know this is the WBC
        league_name = 'World Baseball Classic'

        if away.get('isWinner'):
            champion = away['team']['name']
            runner_up = home['team']['name']
            score = f"{away.get('score')}-{home.get('score')}"
        elif home.get('isWinner'):
            champion = home['team']['name']
            runner_up = away['team']['name']
            score = f"{home.get('score')}-{away.get('score')}"
        else:
            continue

        print(f"{year} [{league_name}]: {champion} (Defeated {runner_up} {score})")

if __name__ == "__main__":
    os.system('cls' if os.name == 'nt' else 'clear')
    get_wbc_champions()

World Baseball Classic Champions:
-----------------------------------------------------------------
2006 [World Baseball Classic]: Japan (Defeated Cuba 10-6)
2009 [World Baseball Classic]: Japan (Defeated Korea 5-3)
2013 [World Baseball Classic]: Dominican Republic (Defeated Puerto Rico 3-0)
2017 [World Baseball Classic]: United States (Defeated Puerto Rico 8-0)
2023 [World Baseball Classic]: Japan (Defeated United States 3-2)
